In [0]:
%sql
DROP TABLE IF EXISTS Employees;
CREATE TABLE Employees (
    emp_id INT PRIMARY KEY,
    name VARCHAR(50),
    department_id INT,
    salary INT,
    manager_id INT
);
DROP TABLE IF EXISTS Departments;
CREATE TABLE Departments (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(50)
);
DROP TABLE IF EXISTS Orders;
CREATE TABLE Orders (
    order_id INT PRIMARY KEY,
    emp_id INT,
    order_amount INT,
    order_date DATE
);

INSERT INTO Employees VALUES
(1, 'Alice', 101, 60000, NULL),
(2, 'Bob', 102, 45000, 1),
(3, 'Charlie', 101, 70000, 1),
(4, 'David', 103, 40000, 2),
(5, 'Eve', 102, 50000, 2);

INSERT INTO Departments VALUES
(101, 'HR'),
(102, 'IT'),
(103, 'Finance');

INSERT INTO Orders VALUES
(1, 1, 5000, '2024-01-10'),
(2, 2, 3000, '2024-02-15'),
(3, 3, 7000, '2024-03-12'),
(4, 1, 2000, '2024-04-01'),
(5, 4, 1000, '2024-05-05');

num_affected_rows,num_inserted_rows
5,5


### Subquery Questions 

### Find employees who earn more than average salary

In [0]:
%sql
select * 
from Employees where salary>(select avg(salary) from Employees)

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


### Find employees who belong to the same department as 'Alice'

In [0]:
%sql
select * from Employees where department_id=(select department_id from Employees where name='Alice')

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


### Get employees whose salary is equal to the minimum salary
 

In [0]:
%sql
select * from Employees 
where salary=(select min(salary) from Employees)

emp_id,name,department_id,salary,manager_id
4,David,103,40000,2


### Find employees who have placed at least one order

In [0]:
%sql
SELECT * FROM Employees e
WHERE EXISTS (SELECT 1 FROM Orders o WHERE o.emp_id = e.emp_id)

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
2,Bob,102,45000,1
3,Charlie,101,70000,1
4,David,103,40000,2


### Get employees whose salary is greater than ALL employees in department 102


In [0]:
%sql
select * from Employees 
where salary > (select max(salary) from Employees where department_id=102)

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


### CTE (Common Table Expression) Questions

### Create a CTE to show employees with salary > 50,000 and fetch all records


In [0]:
%sql
with high_salary_employee as(
    select * from Employees 
    where salary>50000
)
select * from high_salary_employee  

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


### Find average salary per department using CTE

In [0]:
%sql
with average_salary_dept as (
    select department_id,avg(salary) from Employees
    group by department_id
)
select * from average_salary_dept

department_id,avg(salary)
101,65000.0
102,47500.0
103,40000.0


### Use CTE to get employees and their department names



In [0]:
%sql
with emp_dept_name as(
  select e.name,e.department_id,d.dept_name from Employees e
  join Departments d
  on e.department_id=d.dept_id
)
select * from emp_dept_name

name,department_id,dept_name
Alice,101,HR
Bob,102,IT
Charlie,101,HR
David,103,Finance
Eve,102,IT


### Find total order amount per employee using CTE

In [0]:
%sql
with total_order_amount as (
    select emp_id,sum(order_amount) as total_amount
    from orders
    group by emp_id
)
select * from total_order_amount

emp_id,total_amount
1,7000
2,3000
3,7000
4,1000


### Find employees whose salary is above department average using CTE

In [0]:
%sql
WITH dept_avg AS (
    SELECT department_id, AVG(salary) AS avg_salary FROM Employees
    GROUP BY department_id
)
SELECT e.* 
FROM Employees e
JOIN dept_avg d ON e.department_id = d.department_id
WHERE e.salary > d.avg_salary;

emp_id,name,department_id,salary,manager_id
3,Charlie,101,70000,1
5,Eve,102,50000,2
